<a href="https://colab.research.google.com/github/Andres7423/BrazilianEcommerceAnalysis/blob/main/OlistAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
import numpy as np
import pandas as pd

### BRONZE **LAYER**

In [ ]:
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.


In [ ]:
print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/brazilian-ecommerce


In [ ]:
## Import all datasets
customer = pd.read_csv("/kaggle/input/brazilian-ecommerce/olist_customers_dataset.csv")
geolocation =  pd.read_csv("/kaggle/input/brazilian-ecommerce/olist_geolocation_dataset.csv")
items = pd.read_csv("/kaggle/input/brazilian-ecommerce/olist_order_items_dataset.csv")
payments = pd.read_csv("/kaggle/input/brazilian-ecommerce/olist_order_payments_dataset.csv")
reviews = pd.read_csv("/kaggle/input/brazilian-ecommerce/olist_order_reviews_dataset.csv")
orders = pd.read_csv("/kaggle/input/brazilian-ecommerce/olist_orders_dataset.csv")
products = pd.read_csv("/kaggle/input/brazilian-ecommerce/olist_products_dataset.csv")
sellers = pd.read_csv("/kaggle/input/brazilian-ecommerce/olist_sellers_dataset.csv")
ProCategory = pd.read_csv("/kaggle/input/brazilian-ecommerce/product_category_name_translation.csv")


In [ ]:
#Check how many rows and columns each dataset has
customer.shape
geolocation.shape
items.shape
payments.shape
reviews.shape
orders.shape
products.shape
sellers.shape
ProCategory.shape

(71, 2)

In [ ]:
# See the data types
customer.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


# **SILVER LAYER**

In [ ]:
# Check if there are either NaN or Null values
customer.isnull().sum()
geolocation.isnull().sum()
items.isnull().sum()
payments.isnull().sum()
reviews.isna().sum()
orders.isnull().sum()
products.isnull().sum()
sellers.isnull().sum()
ProCategory.isnull().sum()

,0
product_category_name,0
product_category_name_english,0


In [ ]:
customer[customer.duplicated(subset = ['customer_id'], keep = False)].sort_values(by = 'customer_id')
customer.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [ ]:
customer['customer_city'] = customer['customer_city'].str.strip().str.title()

In [ ]:
silver_customer = customer.copy()
silver_customer['customer_id'].value_counts().loc[lambda x:x>1]

,count
customer_id,


## **Items Data Set**

In [ ]:
# Check if there are duplicates
# customer.duplicated(subset = ['customer_id'], keep = False).sum()
#geolocation.duplicated(subset= ['geolocation_zip_code_prefix']).sum()
#items.info()
items['order_id'].value_counts().loc[lambda x: x > 1]

,count
order_id,
8272b63d03f5f79c56e9e4120aec44ef,21
1b15974a0141d54e36626dca3fdc731a,20
ab14fdcfbe524636d65ee38360e22ce8,20
9ef13efd6949e4573a18964dd1bbe7f5,15
428a2f660dc84138d969ccd69a0ab6d5,15
...,...
c629fd3aa69824b58230aee97ab8930d,2
7a5ffe966dbc98f0f43ceeb96e1d73c8,2
4a28fe31d1808616ef3925084765c8b3,2


In [ ]:
items[items['order_id'] == '9ef13efd6949e4573a18964dd1bbe7f5']
items[items.duplicated(keep=False)]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [ ]:
silver_items = items.drop_duplicates('order_id', keep= 'last')

In [ ]:
#Check if there is duplicate values
silver_items['order_id'].value_counts().loc[lambda x: x > 1]

,count
order_id,


In [ ]:
silver_items.dtypes

,0
order_id,object
order_item_id,int64
product_id,object
seller_id,object
shipping_limit_date,object
price,float64
freight_value,float64


## **Payment Data Set**

In [ ]:
payments['order_id'].value_counts().loc[lambda x: x > 1]

,count
order_id,
fa65dad1b0e818e3ccc5cb0e39231352,29
ccf804e764ed5650cd8759557269dc13,26
285c2e15bebd4ac83635ccc563dc71f4,22
895ab968e7bb0d5659d16cd74cd1650c,21
fedcd9f7ccdc8cba3a18defedd1a5547,19
...,...
fccf0d3397491d555ae0d9a4689bfc29,2
65f72c812ba4ca1ebb3c3558c69d5693,2
47cb859a7f526319e3da0015c15c3b1f,2


In [ ]:
# Sort order_id column in order to have the same order id followed by itself
payments = payments.sort_values('order_id')
payments['payment_type'] = payments['payment_type'].str.replace('_', ' ', regex= False).str.strip().str.title()
silver_payments = payments.copy()
silver_payments

,order_id,payment_sequential,payment_type,payment_installments,payment_value
85283,00010242fe8c5a6d1ba2dd792cb16214,1,Credit Card,2,72.19
2499,00018f77f2f0320c557190d7a144bdd3,1,Credit Card,3,259.83
12393,000229ec398224ef6ca0657da4fc703e,1,Credit Card,5,216.87
32971,00024acbcdf0a6daa1e931b038114c75,1,Credit Card,2,25.78
98711,00042b26cf59d7ce69dfabb4e55b4fd9,1,Credit Card,3,218.04
...,...,...,...,...,...
69123,fffc94f6ce00a00581880bf54a75a037,1,Boleto,1,343.40
2776,fffcd46ef2263f404302a634eb57f7eb,1,Boleto,1,386.53
58888,fffce4705a9662cd70adb13d4a31832d,1,Credit Card,3,116.85
37089,fffe18544ffabc95dfada21779c9644f,1,Credit Card,3,64.71


In [ ]:
#How there are some order_id duplicates, order_id and payment_type will be grouped so that we'll get one, two or three lines per order_id and payment_type
silver_payments = silver_payments.groupby(['order_id', 'payment_type'], as_index=False).agg(payment_value = ('payment_value', 'sum'), payment_installments=('payment_installments', 'first'))

In [ ]:
silver_payments = silver_payments.pivot_table(index=['order_id', 'payment_installments'], columns='payment_type', values='payment_value', fill_value=0).reset_index()
silver_payments

payment_type,order_id,payment_installments,Boleto,Credit Card,Debit Card,Not Defined,Voucher
0,00010242fe8c5a6d1ba2dd792cb16214,2,0.00,72.19,0.0,0.0,0.0
1,00018f77f2f0320c557190d7a144bdd3,3,0.00,259.83,0.0,0.0,0.0
2,000229ec398224ef6ca0657da4fc703e,5,0.00,216.87,0.0,0.0,0.0
3,00024acbcdf0a6daa1e931b038114c75,2,0.00,25.78,0.0,0.0,0.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,3,0.00,218.04,0.0,0.0,0.0
...,...,...,...,...,...,...,...
100153,fffc94f6ce00a00581880bf54a75a037,1,343.40,0.00,0.0,0.0,0.0
100154,fffcd46ef2263f404302a634eb57f7eb,1,386.53,0.00,0.0,0.0,0.0
100155,fffce4705a9662cd70adb13d4a31832d,3,0.00,116.85,0.0,0.0,0.0
100156,fffe18544ffabc95dfada21779c9644f,3,0.00,64.71,0.0,0.0,0.0


In [ ]:
silver_payments['order_id'].value_counts().loc[lambda x: x > 2]

,count
order_id,


In [ ]:
silver_payments[silver_payments['order_id'] == '1d3b8d78074e0b50f10346941e67e9de']


payment_type,order_id,payment_installments,Boleto,Credit Card,Debit Card,Not Defined,Voucher
11433,1d3b8d78074e0b50f10346941e67e9de,1,0.0,0.00,0.0,0.0,26.07
11434,1d3b8d78074e0b50f10346941e67e9de,4,0.0,125.23,0.0,0.0,0.00


In [ ]:
silver_payments

payment_type,order_id,payment_installments,Boleto,Credit Card,Debit Card,Not Defined,Voucher
0,00010242fe8c5a6d1ba2dd792cb16214,2,0.00,72.19,0.0,0.0,0.0
1,00018f77f2f0320c557190d7a144bdd3,3,0.00,259.83,0.0,0.0,0.0
2,000229ec398224ef6ca0657da4fc703e,5,0.00,216.87,0.0,0.0,0.0
3,00024acbcdf0a6daa1e931b038114c75,2,0.00,25.78,0.0,0.0,0.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,3,0.00,218.04,0.0,0.0,0.0
...,...,...,...,...,...,...,...
100153,fffc94f6ce00a00581880bf54a75a037,1,343.40,0.00,0.0,0.0,0.0
100154,fffcd46ef2263f404302a634eb57f7eb,1,386.53,0.00,0.0,0.0,0.0
100155,fffce4705a9662cd70adb13d4a31832d,3,0.00,116.85,0.0,0.0,0.0
100156,fffe18544ffabc95dfada21779c9644f,3,0.00,64.71,0.0,0.0,0.0


## **Reviews Data Set**

In [ ]:
reviews.info()
reviews.head(4)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06


In [ ]:
# Standarize the date format
reviews["review_creation_date"] = pd.to_datetime(reviews['review_creation_date'])
reviews.head(2)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13


In [ ]:
#reviews[reviews.duplicated(subset=['order_id'], keep=False)].sort_values(by = 'order_id')
reviews['review_id'].value_counts().loc[lambda x: x>1]

,count
review_id,
4548534449b1f572e357211b90724f1b,3
44e9f871226d8a130de3fc39dfbdf0c5,3
2d6ac45f859465b5c185274a1c929637,3
ddc52555ca27b0fe67d5255147682d2d,3
08528f70f579f0c830189efc523d2182,3
...,...
4687cd3dc8cbaa6b8932ab101d02a937,2
85c3fab7771ced653dbe847f5aeff215,2
dcecf8e3ba584a9ebe49439490a914bb,2


In [ ]:
#Check ow duplicates are working
reviews[reviews['order_id'] == 'ce59496beb223906ab55f6bce7fa86f5']

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
47128,5e336718a575cd6067a6cc14f1f3fb3d,ce59496beb223906ab55f6bce7fa86f5,5,NaN,"A culpa foi dos correios, é pena não termos ou...",2017-06-22,2017-06-26 13:05:25
70337,59294cb7699ef93d259c880d02b00ec1,ce59496beb223906ab55f6bce7fa86f5,5,NaN,NaN,2017-06-17,2017-06-19 20:03:08


In [ ]:
silver_reviews = reviews.drop(columns = ['review_comment_title'])
#silver_reviews = silver_reviews.drop_duplicates(subset='review_id')

In [ ]:
#silver_reviews['review_id'].value_counts().loc[lambda x: x > 1]

# **Orders Data Set**

In [ ]:
orders.info()
orders.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaN,NaN,2017-05-09 00:00:00
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07 00:00:00
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06 00:00:00
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00


In [ ]:
orders[orders.duplicated('order_id', keep= False)].sort_values(by = 'order_id')

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [ ]:
# I'll count the duplicates and loc filters and lambda will do the operation
orders['order_id'].value_counts().loc[lambda x: x>1]

,count
order_id,


In [ ]:
orders['order_status'].unique()

array(['delivered', 'invoiced', 'shipped', 'processing', 'unavailable',
       'canceled', 'created', 'approved'], dtype=object)

In [ ]:
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')

In [ ]:
silver_orders = orders.drop(columns=['order_approved_at', 'order_delivered_carrier_date'])
#The date columns are formatted to datetime
silver_orders[['order_purchase_timestamp', 'order_delivered_customer_date','order_estimated_delivery_date']] = silver_orders[['order_purchase_timestamp','order_delivered_customer_date','order_estimated_delivery_date']].apply(lambda x: pd.to_datetime(x, errors= 'coerce'))
silver_orders.dtypes

,0
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]


In [ ]:
silver_orders.isna().sum()

,0
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_delivered_customer_date,2965
order_estimated_delivery_date,0


## **Products Data Set**

In [ ]:
products.info()
products.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [ ]:
products.loc[:10]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
5,41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60.0,745.0,1.0,200.0,38.0,5.0,11.0
6,732bd381ad09e530fe0a5f457d81becb,cool_stuff,56.0,1272.0,4.0,18350.0,70.0,24.0,44.0
7,2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,56.0,184.0,2.0,900.0,40.0,8.0,40.0
8,37cc742be07708b53a98702e77a21a02,eletrodomesticos,57.0,163.0,1.0,400.0,27.0,13.0,17.0
9,8c92109888e8cdf9d66dc7e463025574,brinquedos,36.0,1156.0,1.0,600.0,17.0,10.0,12.0


In [ ]:
products.isna().sum()

,0
product_id,0
product_category_name,610
product_name_lenght,610
product_description_lenght,610
product_photos_qty,610
product_weight_g,2
product_length_cm,2
product_height_cm,2
product_width_cm,2


In [ ]:
products['product_category_name'] = products['product_category_name'].str.replace('_', ' ', regex=False).str.strip().str.title()

In [ ]:
# Check how many categories there are
products['product_category_name'].unique()

array(['Perfumaria', 'Artes', 'Esporte Lazer', 'Bebes',
       'Utilidades Domesticas', 'Instrumentos Musicais', 'Cool Stuff',
       'Moveis Decoracao', 'Eletrodomesticos', 'Brinquedos',
       'Cama Mesa Banho', 'Construcao Ferramentas Seguranca',
       'Informatica Acessorios', 'Beleza Saude', 'Malas Acessorios',
       'Ferramentas Jardim', 'Moveis Escritorio', 'Automotivo',
       'Eletronicos', 'Fashion Calcados', 'Telefonia', 'Papelaria',
       'Fashion Bolsas E Acessorios', 'Pcs', 'Casa Construcao',
       'Relogios Presentes', 'Construcao Ferramentas Construcao',
       'Pet Shop', 'Eletroportateis', 'Agro Industria E Comercio', nan,
       'Moveis Sala', 'Sinalizacao E Seguranca', 'Climatizacao',
       'Consoles Games', 'Livros Interesse Geral',
       'Construcao Ferramentas Ferramentas',
       'Fashion Underwear E Moda Praia', 'Fashion Roupa Masculina',
       'Moveis Cozinha Area De Servico Jantar E Jardim',
       'Industria Comercio E Negocios', 'Telefonia Fixa',
   

In [ ]:
# How there are products without category I'll create a new dataframe where I store them and a silver dataframe to store the rest
Pnclassified = products[products['product_category_name'].isna() == True]
silver_products = products.dropna(subset = ['product_category_name'])
silver_products = silver_products.drop(columns= ['product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'])

In [ ]:
silver_products.columns

Index(['product_id', 'product_category_name'], dtype='object')

In [ ]:
# We double-check if there are either null or NaN values
silver_products.isnull().sum()

,0
product_id,0
product_category_name,0


## **SELLER DATA SET**

In [ ]:
sellers.info()
sellers.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   object
 1   seller_zip_code_prefix  3095 non-null   int64 
 2   seller_city             3095 non-null   object
 3   seller_state            3095 non-null   object
dtypes: int64(1), object(3)
memory usage: 96.8+ KB


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [ ]:
silver_sellers = sellers.sort_values(by = ['seller_id'])
silver_sellers['seller_city'] = silver_sellers['seller_city'].str.strip().str.title()

In [ ]:
#sellers[sellers.duplicated('seller_id', keep=False)]
#sellers.isnull().sum()
silver_sellers['seller_city'].value_counts().loc[lambda n: n>1]

,count
seller_city,
Sao Paulo,694
Curitiba,127
Rio De Janeiro,96
Belo Horizonte,68
Ribeirao Preto,52
...,...
Coronel Fabriciano,2
Araxa,2
Teresopolis,2


In [ ]:
# Check if the states are lower case
sellers['seller_state'].unique()

array(['SP', 'RJ', 'PE', 'PR', 'GO', 'SC', 'BA', 'DF', 'RS', 'MG', 'RN',
       'MT', 'CE', 'PB', 'AC', 'ES', 'RO', 'PI', 'MS', 'SE', 'MA', 'AM',
       'PA'], dtype=object)

## **Product Category Name Data Set**

In [ ]:
ProCategory.info()
ProCategory.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     object
 1   product_category_name_english  71 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
5,esporte_lazer,sports_leisure
6,perfumaria,perfumery
7,utilidades_domesticas,housewares
8,telefonia,telephony
9,relogios_presentes,watches_gifts


In [ ]:
#Here we pick the columns to remove _ symbol and turn them into title type
ProCategory[['product_category_name', 'product_category_name_english']] = ProCategory[['product_category_name', 'product_category_name_english']].apply(lambda col: col.str.replace("_", " ", regex= False).str.title())
silver_category = ProCategory.copy()
silver_category

,product_category_name,product_category_name_english
0,Beleza Saude,Health Beauty
1,Informatica Acessorios,Computers Accessories
2,Automotivo,Auto
3,Cama Mesa Banho,Bed Bath Table
4,Moveis Decoracao,Furniture Decor
...,...,...
66,Flores,Flowers
67,Artes E Artesanato,Arts And Craftmanship
68,Fraldas Higiene,Diapers And Hygiene
69,Fashion Roupa Infanto Juvenil,Fashion Childrens Clothes


In [ ]:
silver_category['product_category_name'].unique()

array(['Beleza Saude', 'Informatica Acessorios', 'Automotivo',
       'Cama Mesa Banho', 'Moveis Decoracao', 'Esporte Lazer',
       'Perfumaria', 'Utilidades Domesticas', 'Telefonia',
       'Relogios Presentes', 'Alimentos Bebidas', 'Bebes', 'Papelaria',
       'Tablets Impressao Imagem', 'Brinquedos', 'Telefonia Fixa',
       'Ferramentas Jardim', 'Fashion Bolsas E Acessorios',
       'Eletroportateis', 'Consoles Games', 'Audio', 'Fashion Calcados',
       'Cool Stuff', 'Malas Acessorios', 'Climatizacao',
       'Construcao Ferramentas Construcao',
       'Moveis Cozinha Area De Servico Jantar E Jardim',
       'Construcao Ferramentas Jardim', 'Fashion Roupa Masculina',
       'Pet Shop', 'Moveis Escritorio', 'Market Place', 'Eletronicos',
       'Eletrodomesticos', 'Artigos De Festas', 'Casa Conforto',
       'Construcao Ferramentas Ferramentas', 'Agro Industria E Comercio',
       'Moveis Colchao E Estofado', 'Livros Tecnicos', 'Casa Construcao',
       'Instrumentos Musicais', 'M

In [ ]:
silver_category[silver_category.duplicated('product_category_name', keep = False)]

,product_category_name,product_category_name_english


In [ ]:
silver_orders['order_status'].unique()

array(['delivered', 'invoiced', 'shipped', 'processing', 'unavailable',
       'canceled', 'created', 'approved'], dtype=object)

## **JOIN DATA TO CREATE A CONSOLIDATED DATASET**

In [ ]:
masterOrders = silver_orders.merge(silver_items, on = 'order_id', how = 'inner')
masterOrders = masterOrders.merge(silver_products, on = 'product_id', how = 'inner')
masterOrders = masterOrders.merge(silver_customer, on = 'customer_id', how = 'inner')
masterOrders = masterOrders.merge(silver_payments, on = 'order_id', how = 'inner')
masterOrders = masterOrders[masterOrders['order_status'] == 'delivered']

In [ ]:
#It is necessary to stretch out the width
pd.set_option('display.width', None)
pd.set_option('display.max_columns', None)

In [ ]:
silver_products

,product_id,product_category_name
0,1e9e8ef04dbcff4541ed26657ea517e5,Perfumaria
1,3aa071139cb16b67ca9e5dea641aaa2f,Artes
2,96bd76ec8810374ed1b65e291975717f,Esporte Lazer
3,cef67bcfe19066a932b7673e239eb23d,Bebes
4,9dc1a7de274444849c219cff195d0b71,Utilidades Domesticas
...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,Moveis Decoracao
32947,bf4538d88321d0fd4412a93c974510e6,Construcao Ferramentas Iluminacao
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,Cama Mesa Banho
32949,83808703fc0706a22e264b9d75f04a2e,Informatica Acessorios


In [ ]:
masterSeller = silver_items.merge(silver_sellers, on = 'seller_id', how='inner')
masterSeller = masterSeller.merge(silver_orders, on = 'order_id', how = 'inner')
masterSeller = masterSeller.merge(silver_reviews, on='order_id', how = 'inner')
masterSeller = masterSeller.merge(silver_products, on= 'product_id', how = 'inner')
masterSeller.columns

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value',
       'seller_zip_code_prefix', 'seller_city', 'seller_state', 'customer_id',
       'order_status', 'order_purchase_timestamp',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'review_id', 'review_score', 'review_comment_message',
       'review_creation_date', 'review_answer_timestamp',
       'product_category_name'],
      dtype='object')

In [ ]:
masterSeller = masterSeller.sort_values(by = 'review_answer_timestamp', ascending= False)

In [ ]:
masterSeller = masterSeller.drop_duplicates(subset='order_id', keep='first')

In [ ]:
masterSeller['order_id'].duplicated().sum()

np.int64(0)

In [ ]:
masterOrders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date',
       'price', 'freight_value', 'product_category_name', 'customer_unique_id',
       'customer_zip_code_prefix', 'customer_city', 'customer_state',
       'payment_installments', 'Boleto', 'Credit Card', 'Debit Card',
       'Not Defined', 'Voucher'],
      dtype='object')

## **GOLDEN LAYER**

**REVENUES**

In [ ]:
revenue = masterOrders.drop(columns= ['order_item_id', 'shipping_limit_date', 'order_estimated_delivery_date'])

In [ ]:
# Using the melt function I'll unpivot payment types and its values so that we can get the revenue later on
revenue = revenue.melt(
    id_vars=[
        'customer_id','customer_unique_id','order_id', 'order_status', 'order_purchase_timestamp', 'product_id',
        'seller_id', 'customer_zip_code_prefix', 'customer_city', 'product_category_name',
        'customer_state', 'payment_installments','freight_value','price'
    ],
    value_vars=[
        'Boleto', 'Credit Card', 'Debit Card', 'Not Defined', 'Voucher'
    ],
    var_name='PaymentType',
    value_name='Value'
)

In [ ]:
revenue['Month_Year'] = revenue['order_purchase_timestamp'].dt.to_period('M')

In [ ]:
revenue

,customer_id,customer_unique_id,order_id,order_status,order_purchase_timestamp,product_id,seller_id,customer_zip_code_prefix,customer_city,product_category_name,customer_state,payment_installments,freight_value,price,PaymentType,Value,Month_Year
0,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-02 10:56:33,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,3149,Sao Paulo,Utilidades Domesticas,SP,1,8.72,29.99,Boleto,0.00,2017-10
1,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07-24 20:41:37,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,47813,Barreiras,Perfumaria,BA,1,22.76,118.70,Boleto,141.46,2018-07
2,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-08 08:38:49,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,75265,Vianopolis,Automotivo,GO,3,19.22,159.90,Boleto,0.00,2018-08
3,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11-18 19:28:06,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,59296,Sao Goncalo Do Amarante,Pet Shop,RN,1,27.20,45.00,Boleto,0.00,2017-11
4,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-13 21:18:39,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,9195,Santo Andre,Papelaria,SP,1,8.72,19.90,Boleto,0.00,2018-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
479030,39bd1228ee8140590ac3aca26f2dfe00,6359f309b166b0196dbf7ad2ac62bb5a,9c5dedf39a927c1b2549525ed64a053c,delivered,2017-03-09 09:54:05,ac35486adb7b02598c182c2ff2e05254,e24fc9fcd865784fb25705606fe3dfe7,12209,Sao Jose Dos Campos,Beleza Saude,SP,3,13.08,72.00,Voucher,0.00,2017-03
479031,1fca14ff2861355f6e5f14306ff977a7,da62f9e57a76d978d02ab5362c509660,63943bddc261676b46f01ca7ac2f7bd8,delivered,2018-02-06 12:58:58,f1d4ce8c6dd66c47bbaa8c6781c2a923,1f9ab4708f3056ede07124aad39a2554,11722,Praia Grande,Bebes,SP,3,20.10,174.90,Voucher,0.00,2018-02
479032,1aa71eb042121263aafbe80c1b562c9c,737520a9aad80b3fbbdad19b66b37b30,83c1379a015df1e13d02aae0204711ab,delivered,2017-08-27 14:46:43,b80910977a37536adeddd63663f916ad,d50d79cb34e38265a8649c383dcffd48,45920,Nova Vicosa,Eletrodomesticos 2,BA,5,65.02,205.99,Voucher,0.00,2017-08
479033,b331b74b18dc79bcdf6532d51e1637c1,5097a5312c8b157bb7be58ae360ef43c,11c177c8e97725db2631073c19f07b62,delivered,2018-01-08 21:28:27,d1c427060a0f73f6b889a5c7c61f2ac4,a1043bafd471dff536d0c462352beb48,28685,Japuiba,Informatica Acessorios,RJ,4,40.59,179.99,Voucher,0.00,2018-01


In [ ]:
revenueMonth = revenue.groupby('Month_Year', as_index=False).agg(Total = ('Value', 'sum')).sort_values(by = 'Month_Year')

**REVENUE PER MONTH AND YEAR**

In [ ]:
revenueMonth['M_Y'] = revenueMonth['Month_Year'].dt.strftime('%B %Y')
revenueMonth

,Month_Year,Total,M_Y
0,2016-10,46470.79,October 2016
1,2016-12,19.62,December 2016
2,2017-01,125746.94,January 2017
3,2017-02,261752.64,February 2017
4,2017-03,407297.23,March 2017
5,2017-04,381077.18,April 2017
6,2017-05,554610.48,May 2017
7,2017-06,484426.47,June 2017
8,2017-07,562628.48,July 2017
9,2017-08,639789.34,August 2017


**REVENUE PER YEAR**

In [ ]:
revenueYear = revenue.groupby(revenue['order_purchase_timestamp'].dt.to_period('Y')).agg(TotalRevenuePerYear = ('Value', 'sum')).reset_index()
revenueYear.rename(columns={'order_purchase_timestamp': 'Year'})

,Year,TotalRevenuePerYear
0,2016,46490.41
1,2017,6808674.97
2,2018,8370323.27


**REVENUE BY STATE**

In [ ]:
#Check out unique values
revenue['customer_state'].unique()

array(['SP', 'BA', 'GO', 'RN', 'PR', 'RJ', 'RS', 'MG', 'SC', 'RR', 'PE',
       'TO', 'CE', 'DF', 'SE', 'MT', 'PB', 'PA', 'RO', 'ES', 'AP', 'MS',
       'MA', 'PI', 'AL', 'AC', 'AM'], dtype=object)

In [ ]:
revenueState = revenue.groupby('customer_state').agg(RevenueState = ('Value', 'sum')).reset_index()
revenueState

,customer_state,RevenueState
0,AC,19458.91
1,AL,93505.35
2,AM,26957.52
3,AP,16141.81
4,BA,583718.42
5,CE,262917.95
6,DF,342990.53
7,ES,315811.36
8,GO,324811.08
9,MA,146223.36


**REVENUE BY PRODUCT CATEGORY**

In [ ]:
revenueCategory = revenue.groupby('product_category_name').agg(RevenueNameCategory = ('Value', 'sum')).reset_index()
revenueCategory

,product_category_name,RevenueNameCategory
0,Agro Industria E Comercio,76208.42
1,Alimentos,36152.97
2,Alimentos Bebidas,19571.71
3,Artes,27524.07
4,Artes E Artesanato,2204.07
...,...,...
68,Sinalizacao E Seguranca,27695.36
69,Tablets Impressao Imagem,9093.57
70,Telefonia,380896.96
71,Telefonia Fixa,59601.04


**REVENUE PER CUSTOMER**

In [ ]:
revenueCustomer = revenue.groupby('customer_unique_id', as_index=False).agg(TotalPerCustomer = ('Value', 'sum'))
revenueCustomer

,customer_unique_id,TotalPerCustomer
0,0000366f3b9a7992bf8c76cfdf3221e2,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,27.19
2,0000f46a3911fa3c0805444483337064,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,43.62
4,0004aac84e0df4da2b147fca70cf8255,196.89
...,...,...
92066,fffcf5a5ff07b0908bd4e2dbc735a684,2067.42
92067,fffea47cd6d3cc0a88bd621562a9d061,84.58
92068,ffff371b4d645b6ecea244b27531430a,112.46
92069,ffff5962728ec6157033ef9805bacc48,133.69


**TOTAL ORDERS**

In [ ]:
Total_Orders = masterOrders['order_id'].nunique()

print(f'Total Orders: {Total_Orders}')

Total Orders: 95118


**TOTAL SPEND BY CUSTOMER**

In [ ]:
TotalSpendbyCustomer = revenue.groupby(['customer_unique_id', 'freight_value'], as_index=False).agg(Total = ('Value', 'sum'))
TotalSpendbyCustomer['Total_Order'] = TotalSpendbyCustomer[['freight_value', 'Total']].sum(axis=1)
TotalSpendbyCustomer

,customer_unique_id,freight_value,Total,Total_Order
0,0000366f3b9a7992bf8c76cfdf3221e2,12.00,141.90,153.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,8.29,27.19,35.48
2,0000f46a3911fa3c0805444483337064,17.22,86.22,103.44
3,0000f6ccb0745a6a4b88665a16c9f078,17.63,43.62,61.25
4,0004aac84e0df4da2b147fca70cf8255,16.89,196.89,213.78
...,...,...,...,...
94667,fffcf5a5ff07b0908bd4e2dbc735a684,239.14,2067.42,2306.56
94668,fffea47cd6d3cc0a88bd621562a9d061,19.69,84.58,104.27
94669,ffff371b4d645b6ecea244b27531430a,22.56,112.46,135.02
94670,ffff5962728ec6157033ef9805bacc48,18.69,133.69,152.38


**AVERAGE ORDER VALUE**

In [ ]:
price = masterOrders.groupby('order_id', as_index=False).agg({'price': 'first'})

In [ ]:
total_revenue = price['price'].sum()
print(f'Total revenue {total_revenue}')

Total revenue 11919342.14


In [ ]:
AOV = total_revenue/Total_Orders
print(f'The average order value is ${round(AOV)}')

The average order value is $125


**RECENCY**

In [ ]:
#I grouped it because there were more rows per order id due t payment methods
Recency = masterOrders.groupby(['order_id', 'customer_unique_id'], as_index=False).agg(lastpurchasedate = ('order_purchase_timestamp', 'max')).sort_values(by='lastpurchasedate', ascending=False)
Recency

,order_id,customer_unique_id,lastpurchasedate
19870,35a972d7f8436f405b56e36add1a7140,24ac2b4327e25baf39f2119e4228976a,2018-08-29 15:00:37
1467,03ef5dedbe7492bdae72eec50764c43f,b701bebbdf478f5500348f03aff62121,2018-08-29 14:52:00
8345,168626408cb32af0ffaf76711caae1dc,7febafa06d9d8f232a900a2937f04338,2018-08-29 14:18:28
4187,0b223d92c27432930dfe407c6aea3041,5c58de6fb80e93396e2f35642666b693,2018-08-29 14:18:23
30501,52018484704db3661b98ce838612b507,7a22d14aa3c3599238509ddca4b93b01,2018-08-29 12:25:59
...,...,...,...
76300,cd3b8574c82b42fc8129f6d502690c3e,87776adb449c551e74c13fc34f036105,2016-10-03 22:31:31
78151,d207cc272675637bfed0062edffd0818,8d3a54507421dbd2ce0a1d58046826e0,2016-10-03 22:06:03
60690,a41c8759fbe7aab36ea07e038b2d4465,61db744d2f835035a5625b59350c6b63,2016-10-03 21:13:36
70730,be5bc2f0da14d8071e2d45451ad119d9,2f64e403852e6893ae37485d5fcacdaf,2016-10-03 16:56:50


In [ ]:
# To check how it is its behaviour
Recency[Recency['customer_unique_id']== '8d50f5eadf50201ccdcedfb9e2ac8455']

,order_id,customer_unique_id,lastpurchasedate
78718,d3582fd5ccccd9cb229a63dfb417c86f,8d50f5eadf50201ccdcedfb9e2ac8455,2018-08-20 19:14:26
59781,a1a6eadae352582010f449d8527b55f3,8d50f5eadf50201ccdcedfb9e2ac8455,2018-08-18 12:39:19
72148,c2213109a2cc0e75d55585b7aaac6d97,8d50f5eadf50201ccdcedfb9e2ac8455,2018-08-07 23:32:14
6450,112eb6f37f1b9dabbced368fbbc6c9ef,8d50f5eadf50201ccdcedfb9e2ac8455,2018-07-23 21:53:02
48751,838f8e96cf584a1d8d22eaf7143c7772,8d50f5eadf50201ccdcedfb9e2ac8455,2018-07-04 22:35:13
40043,6bdf325f0966e3056651285c0aed5aad,8d50f5eadf50201ccdcedfb9e2ac8455,2018-05-22 23:08:55
13065,23427a6bd9f8fd1b51f1b1e5cc186ab8,8d50f5eadf50201ccdcedfb9e2ac8455,2018-05-21 22:44:31
68437,b850a16d8faf65a74c51287ef34379ce,8d50f5eadf50201ccdcedfb9e2ac8455,2017-11-22 20:01:53
51037,89d9b111d2b990deb5f5f9769f92800b,8d50f5eadf50201ccdcedfb9e2ac8455,2017-10-29 16:58:02
76324,cd4b336a02aacabd0ef22f6db711f95e,8d50f5eadf50201ccdcedfb9e2ac8455,2017-10-18 23:25:04


In [ ]:
# I grouped agin to get only a row per customer and I used shift(-1) to go up the take the prior date
Recency['PriorDate']  = Recency.groupby('customer_unique_id', as_index=False)['lastpurchasedate'].shift(-1)

In [ ]:
# For recency, we'd put a condition for each row and we are going to substract the current date if there is no prior last purchase
Recency['DaysBetweenPurchases'] = Recency[['lastpurchasedate', 'PriorDate']].apply(lambda x: pd.Timestamp.now() - x['lastpurchasedate'] if pd.isna(x['PriorDate']) else x['lastpurchasedate'] - x['PriorDate'], axis=1)
Recency['Recency'] = pd.Timestamp.today() - Recency['lastpurchasedate']


In [ ]:
# Days are formatted to get only how many days have passed
Recency['DaysBetweenPurchases'] = Recency['DaysBetweenPurchases'].dt.days

In [ ]:
Recency['Recency'] = Recency['Recency'].dt.days

In [ ]:
Recency[Recency['customer_unique_id']== '8d50f5eadf50201ccdcedfb9e2ac8455']

,order_id,customer_unique_id,lastpurchasedate,PriorDate,DaysBetweenPurchases,Recency
78718,d3582fd5ccccd9cb229a63dfb417c86f,8d50f5eadf50201ccdcedfb9e2ac8455,2018-08-20 19:14:26,2018-08-18 12:39:19,2,2962
59781,a1a6eadae352582010f449d8527b55f3,8d50f5eadf50201ccdcedfb9e2ac8455,2018-08-18 12:39:19,2018-08-07 23:32:14,10,2964
72148,c2213109a2cc0e75d55585b7aaac6d97,8d50f5eadf50201ccdcedfb9e2ac8455,2018-08-07 23:32:14,2018-07-23 21:53:02,15,2974
6450,112eb6f37f1b9dabbced368fbbc6c9ef,8d50f5eadf50201ccdcedfb9e2ac8455,2018-07-23 21:53:02,2018-07-04 22:35:13,18,2989
48751,838f8e96cf584a1d8d22eaf7143c7772,8d50f5eadf50201ccdcedfb9e2ac8455,2018-07-04 22:35:13,2018-05-22 23:08:55,42,3008
40043,6bdf325f0966e3056651285c0aed5aad,8d50f5eadf50201ccdcedfb9e2ac8455,2018-05-22 23:08:55,2018-05-21 22:44:31,1,3051
13065,23427a6bd9f8fd1b51f1b1e5cc186ab8,8d50f5eadf50201ccdcedfb9e2ac8455,2018-05-21 22:44:31,2017-11-22 20:01:53,180,3052
68437,b850a16d8faf65a74c51287ef34379ce,8d50f5eadf50201ccdcedfb9e2ac8455,2017-11-22 20:01:53,2017-10-29 16:58:02,24,3232
51037,89d9b111d2b990deb5f5f9769f92800b,8d50f5eadf50201ccdcedfb9e2ac8455,2017-10-29 16:58:02,2017-10-18 23:25:04,10,3257
76324,cd4b336a02aacabd0ef22f6db711f95e,8d50f5eadf50201ccdcedfb9e2ac8455,2017-10-18 23:25:04,2017-09-05 22:14:52,43,3267


## **CUSTOMER SEGMENTS**

In [ ]:
# I will segment the customer according to its loyalty
bins = [0, 2, 5, 20]
labels = ['Occasional', 'Loyal', 'VIP']
Loyalty = Recency.groupby('customer_unique_id', as_index= False).agg(NumberofPurchase = ('order_id', 'count'))

In [ ]:
Loyalty['Loyalty'] = pd.cut(Loyalty['NumberofPurchase'], bins= bins, labels=labels)
Loyalty['NumberofPurchase'].value_counts()

,count
NumberofPurchase,
1,89330
2,2520
3,175
4,29
5,9
6,3
7,3
9,1
14,1


## **DELIVERY PERFORMANCE**

In [ ]:
masterOrders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date',
       'price', 'freight_value', 'product_category_name', 'customer_unique_id',
       'customer_zip_code_prefix', 'customer_city', 'customer_state',
       'payment_installments', 'Boleto', 'Credit Card', 'Debit Card',
       'Not Defined', 'Voucher'],
      dtype='object')

In [ ]:
silver_orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26
...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-17 15:08:01,2017-03-28
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-28 17:37:56,2018-03-02
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-09-21 11:24:17,2017-09-27
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-25 23:32:54,2018-02-15


In [ ]:
Delivery = masterOrders.drop(columns=['payment_installments', 'Boleto', 'Credit Card', 'Debit Card',
       'Not Defined', 'Voucher','order_item_id', 'product_id', 'customer_zip_code_prefix', 'price', 'freight_value'])
Delivery.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,seller_id,shipping_limit_date,product_category_name,customer_unique_id,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,Utilidades Domesticas,7c396fd4830fd04220f754e42b4e5bff,Sao Paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,Perfumaria,af07308b275d755c9edb36a90c618231,Barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,Automotivo,3a653a41f6f9fc3d2a113cf8398680e8,Vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,Pet Shop,7c142cf63193a1473d2e66489a9ae977,Sao Goncalo Do Amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,Papelaria,72632f0f9dd73dfee390c9b22eb56dd6,Santo Andre,SP


In [ ]:
Delivery['DelayDays'] = Delivery['order_estimated_delivery_date'] - Delivery['order_delivered_customer_date']
Delivery['CurrentStatus'] = np.where(Delivery['DelayDays'] > pd.Timedelta(days=0), 'On time', 'Delayed')

In [ ]:
Delivery['DelayDays'] = Delivery['DelayDays'].dt.days

In [ ]:
Delivery['order_id'].value_counts().loc[lambda x: x > 1]

,count
order_id,
236909a5eec961f916c742c7f991d3e5,2
0e3ec3c17b64c6cc7bc01bba21e7afdf,2
4e376d36ceeae3f2cbf66cb0ca84c799,2
fd79284ba0bd11e560e236f939640143,2
03c968d10d8c63bd2388245d02579fde,2
...,...
44001fa5bde42ef2bb752eab4c9dde60,2
70009c6d985dfa16754461cff33c10dc,2
187c20e7849e29f8582a8945283a6f01,2


In [ ]:
Delivery = Delivery.drop_duplicates(subset = 'order_id')

In [ ]:
Delivery.dtypes

,0
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]
seller_id,object
shipping_limit_date,object
product_category_name,object
customer_unique_id,object


In [ ]:
counts = Delivery['CurrentStatus'].value_counts()

In [ ]:
Ontime = counts['On time']

In [ ]:
Delay = counts['Delayed']

In [ ]:
print(f'Number of orders on time is: {Ontime}')
print(f'Number of orders on time is: {Delay}')

Number of orders on time is: 87409
Number of orders on time is: 7709


## **SELLER PERFORMANCE**

In [ ]:
masterSeller['order_status'].unique()

array(['delivered', 'canceled', 'invoiced', 'shipped', 'processing',
       'unavailable', 'approved'], dtype=object)

In [ ]:
masterSeller[masterSeller['order_status'] == 'processing']

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,seller_zip_code_prefix,seller_city,seller_state,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_message,review_creation_date,review_answer_timestamp,product_category_name
52804,8bc9548dbe844e1bf48ac197c5609045,2,1d7db62a21ffb4f220d3e8a05fa77e02,643214e62b870443ccbe55ab29a4dccf,2018-07-26 09:17:48,32.00,25.45,2134,Sao Paulo,SP,d4dc57fd18dfe3e30be4d066d873d388,processing,2018-07-23 18:03:03,NaT,2018-08-13,c0a92e2b5ef5016dc2ae8220f83de562,5,NaN,2018-08-15,2018-08-15 09:18:15,Fashion Bolsas E Acessorios
33261,57c432fd96e9d52639f048eb1b2a5b10,1,83b00325c13c44245b2c3a2befa62a0e,ad781527c93d00d89a11eecd9dcad7c1,2018-05-24 04:31:20,139.99,36.84,15015,Sao Jose Do Rio Preto,SP,fdd0411bec46e2d9a15590016b897dc3,processing,2018-05-07 19:30:29,NaT,2018-06-04,e3612b7b9f5872e01bbddd0ff19f184c,3,comprei kit de berco a mais de um mes e ainda ...,2018-06-08,2018-06-11 15:34:44,Bebes
49907,83fb83d45702e06d5d5ddbcfd0b8d2a0,1,04d5a0ec1f6db00db01676199151df4c,fa1c13f2614d7b5c4749cbc52fecda94,2018-05-07 02:31:03,139.90,23.48,13170,Sumare,SP,25d32598e801258a7a8ade492cf3a836,processing,2018-04-30 19:08:15,NaT,2018-06-06,9795b5fb0094ddd06f7688dab1f5bb26,1,Fiz o pedido em 30-04-2018 e paguei no mesmo d...,2018-06-09,2018-06-09 13:50:31,Relogios Presentes
48884,8124e0a6295df5f9ce4377ca0a8e0c18,1,83b00325c13c44245b2c3a2befa62a0e,ad781527c93d00d89a11eecd9dcad7c1,2018-05-24 17:36:39,139.99,36.84,15015,Sao Jose Do Rio Preto,SP,6e14fc1f239d4384a91b043639b2e3b7,processing,2018-05-08 15:27:57,NaT,2018-06-05,5468015c5083e6d6f3c9e3f8b196cd52,1,"Era pra ter chego até o dia 06/06/2018, já fui...",2018-06-08,2018-06-08 12:26:21,Bebes
49908,83fc33b62b8c7c39e2258d081955143d,1,c0abb5707b6d57b4e7d9797222a77fc8,bc07d855eac23aab5ac8a120f21d16b2,2018-05-28 02:55:42,80.49,8.33,8710,Mogi Das Cruzes,SP,c6eb22e4e7b7e5b871c06250c3296ef9,processing,2018-05-15 20:00:48,NaT,2018-05-30,54a8ac23bac5c6a2cb18df53795af0a6,1,Ainda não recebi meu produto,2018-06-06,2018-06-08 04:18:08,Beleza Saude
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31573,533e104c4193cf07d264603f7800fb71,1,f142a5ca78eb112cafbcf29f9a43eca7,02a2272692e13558373c66db98f05e2e,2017-02-10 15:33:57,29.99,14.52,20040,Rio De Janeiro,RJ,2f95fc8f86679d432d77e9d0316fac48,processing,2017-02-06 15:33:57,NaT,2017-03-03,af37e86291b016d85be4b78f7782be5c,1,Nao recebi o produto,2017-03-05,2017-03-07 04:32:40,Telefonia
14098,255d951095fc26f86cc7a08b822b714a,1,0d1d766e7cdd4a724e9ea9a18e5177bb,75d34ebb1bd0bd7dde40dd507b8169c3,2017-01-19 15:18:25,755.99,26.75,41820,Salvador,BA,4aa40fd3991ea64e9dead06fe15f021c,processing,2017-01-11 15:18:25,NaT,2017-03-01,4d038732e45b5c55a5d0f5c7b7ddd9b2,1,Só quero que a lannister resolva esse problema...,2017-03-03,2017-03-06 23:15:59,Telefonia
42008,6ed4f19dc97f2a4b5d0f15651247dbca,1,3948f284081821f4e27ba8c23d4cbc9a,c542f5a8c64d973f30e9df8d06f70b4a,2017-01-22 09:52:32,81.00,17.31,2013,Sao Paulo,SP,6b7e649105e4c5388bd8c3c4e9efcff3,processing,2017-01-18 09:52:32,NaT,2017-02-28,4d0f5135f9a2024943166671431bd897,1,Péssimo atendimento! Comprei o produto no iníc...,2017-03-02,2017-03-02 15:33:00,Brinquedos
32948,56eb381fff8db8d35134693062514084,1,7f524cd9e1d47c5aa4b15b54d6d146cb,7a073a40a9d3f1fee8bc45670ea1e909,2016-10-25 06:03:26,95.00,17.78,86040,Londrina,PR,b102a37db53345d7eb418649b096f8e8,processing,2016-10-09 12:34:53,NaT,2016-12-30,f7c4ece51a1f0f3ebf9a67db7df64036,1,Nenhum dos produtos que comprei desta loja vie...,2017-01-04,2017-01-04 09:09:23,Moveis Decoracao


In [ ]:
masterSeller.columns

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value',
       'seller_zip_code_prefix', 'seller_city', 'seller_state', 'customer_id',
       'order_status', 'order_purchase_timestamp',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'review_id', 'review_score', 'review_comment_message',
       'review_creation_date', 'review_answer_timestamp',
       'product_category_name'],
      dtype='object')

In [ ]:
sellerPerformance = masterSeller.drop(columns=['shipping_limit_date', 'product_id',
       'seller_zip_code_prefix', 'order_purchase_timestamp'])

**REVENUE**

In [ ]:
sellerPerformance['TotalPaid'] = sellerPerformance[['freight_value', 'price']].sum(axis=1)
sellerRevenue = sellerPerformance[sellerPerformance['order_status'] == 'delivered']

In [ ]:
sellerRevenue = sellerRevenue.groupby('seller_id', as_index=False).agg(Revenue = ('TotalPaid', 'sum'))

In [ ]:
sellerRevenue['Rank'] = sellerRevenue['Revenue'].rank(ascending=False)

In [ ]:
sellerRevenue = sellerRevenue.sort_values('Revenue', ascending=False)
sellerRevenue.head(5)

,seller_id,Revenue,Rank
817,4869f7a5dfa277a7dca6462dcf3b52b2,239493.11,1.0
962,53243585a1d6dc2643021fd1853d8905,212737.26,2.0
841,4a3ca9315b744ce9f8e9374361493884,203047.44,3.0
2832,fa1c13f2614d7b5c4749cbc52fecda94,198208.82,4.0
1445,7c67e1448b00f6e969d365cea6b010ab,173029.07,5.0


 **AVERAGE REVIEW SCORE**

In [ ]:
avgSeller = masterSeller.groupby('seller_id', as_index=False).agg(Score = ('review_score', 'mean'))
avgSeller = avgSeller.sort_values(by= 'Score', ascending=False)
avgSeller

,seller_id,Score
2987,fe19dce63ae80346207c6c55713d1023,5.0
2986,fdf736c18c589ed030e058312203e1b2,5.0
2982,fd435faa3c0422b60440ea3480d0e77c,5.0
2980,fd312b6bf05efac6c3772d5b52205d8a,5.0
21,01ed254b9ff8407dfb9d99ba1e17d923,5.0
...,...,...
1317,6e85dc5ecd97a61094b89b046a509d8e,1.0
2971,fc6295add6f51a0936407ead70c1001d,1.0
2974,fc99f99385ae90f66f01733856fb88c9,1.0
11,010da0602d7774602cd1b3f5fb7b709e,1.0


**NUMBER OF ORDERS**

In [ ]:
orderSellers = masterSeller.groupby('seller_id',as_index=False).agg(nOrders = ('order_id', 'count'))
orderSellers = orderSellers.sort_values(by='nOrders', ascending=False)
orderSellers

,seller_id,nOrders
1208,6560211a19b47992c3666cc44a7e94c0,1819
861,4a3ca9315b744ce9f8e9374361493884,1736
2418,cc419e0650a3c5ba77189a1882b7556a,1696
361,1f50f920176fa81dab994f9023523100,1380
2578,da8622b14eb17ae2831f4ac5b9dab84a,1289
...,...,...
2974,fc99f99385ae90f66f01733856fb88c9,1
2971,fc6295add6f51a0936407ead70c1001d,1
2968,fc4751cfe3b279c419615b9b00c1abb4,1
1470,7a704e63bb29c446869efde5d57524c8,1


# **TOP CATEGORIES**

In [ ]:
topCate = masterOrders.groupby('product_category_name', as_index=False).agg(NumberOrders = ('product_category_name', 'count'))

In [ ]:
topCate['Rank'] = topCate['NumberOrders'].rank(ascending=False)

In [ ]:
topCate = topCate.sort_values(by='Rank')

In [ ]:
topCate.head(10)

,product_category_name,NumberOrders,Rank
13,Cama Mesa Banho,9253,1.0
11,Beleza Saude,8657,2.0
32,Esporte Lazer,7550,3.0
44,Informatica Acessorios,6554,4.0
54,Moveis Decoracao,6247,5.0
72,Utilidades Domesticas,5723,6.0
66,Relogios Presentes,5532,7.0
70,Telefonia,4108,8.0
8,Automotivo,3810,9.0
12,Brinquedos,3800,10.0


# **Fastest growing categories**


In [ ]:
growingcate = revenue.groupby(['Month_Year', 'product_category_name'], as_index=False).agg(norderovertime = ('order_id', 'count'))

In [ ]:
growingcate = growingcate.pivot_table(index=['Month_Year'], columns='product_category_name', values='norderovertime', fill_value=0)

In [ ]:
growingcate

product_category_name,Agro Industria E Comercio,Alimentos,Alimentos Bebidas,Artes,Artes E Artesanato,Artigos De Festas,Artigos De Natal,Audio,Automotivo,Bebes,Bebidas,Beleza Saude,Brinquedos,Cama Mesa Banho,Casa Conforto,Casa Conforto 2,Casa Construcao,Cds Dvds Musicais,Cine Foto,Climatizacao,Consoles Games,Construcao Ferramentas Construcao,Construcao Ferramentas Ferramentas,Construcao Ferramentas Iluminacao,Construcao Ferramentas Jardim,Construcao Ferramentas Seguranca,Cool Stuff,Dvds Blu Ray,Eletrodomesticos,Eletrodomesticos 2,Eletronicos,Eletroportateis,Esporte Lazer,Fashion Bolsas E Acessorios,Fashion Calcados,Fashion Esporte,Fashion Roupa Feminina,Fashion Roupa Infanto Juvenil,Fashion Roupa Masculina,Fashion Underwear E Moda Praia,Ferramentas Jardim,Flores,Fraldas Higiene,Industria Comercio E Negocios,Informatica Acessorios,Instrumentos Musicais,La Cuisine,Livros Importados,Livros Interesse Geral,Livros Tecnicos,Malas Acessorios,Market Place,Moveis Colchao E Estofado,Moveis Cozinha Area De Servico Jantar E Jardim,Moveis Decoracao,Moveis Escritorio,Moveis Quarto,Moveis Sala,Musica,Papelaria,Pc Gamer,Pcs,Perfumaria,Pet Shop,Portateis Casa Forno E Cafe,Portateis Cozinha E Preparadores De Alimentos,Relogios Presentes,Seguros E Servicos,Sinalizacao E Seguranca,Tablets Impressao Imagem,Telefonia,Telefonia Fixa,Utilidades Domesticas
Month_Year,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2016-10,0.0,5.0,0.0,0.0,0.0,0.0,0.0,10.0,40.0,45.0,0.0,180.0,110.0,15.0,0.0,0.0,0.0,0.0,0.0,20.0,35.0,0.0,0.0,0.0,0.0,0.0,35.0,0.0,0.0,0.0,5.0,0.0,85.0,35.0,5.0,0.0,5.0,0.0,5.0,0.0,25.0,0.0,5.0,15.0,60.0,0.0,0.0,0.0,0.0,0.0,0.0,25.0,0.0,0.0,250.0,30.0,0.0,0.0,0.0,0.0,0.0,0.0,130.0,20.0,0.0,0.0,20.0,0.0,0.0,0.0,35.0,25.0,45.0
2016-12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2017-01,10.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,135.0,155.0,0.0,400.0,205.0,215.0,0.0,10.0,0.0,0.0,0.0,15.0,95.0,0.0,0.0,0.0,0.0,0.0,155.0,5.0,0.0,30.0,55.0,30.0,295.0,155.0,5.0,0.0,10.0,0.0,5.0,10.0,190.0,0.0,0.0,0.0,120.0,5.0,0.0,0.0,10.0,0.0,20.0,25.0,0.0,5.0,650.0,30.0,40.0,40.0,10.0,35.0,0.0,5.0,150.0,60.0,0.0,0.0,55.0,5.0,0.0,0.0,85.0,95.0,105.0
2017-02,35.0,40.0,0.0,0.0,0.0,0.0,0.0,25.0,375.0,150.0,0.0,715.0,310.0,700.0,15.0,0.0,15.0,0.0,5.0,55.0,90.0,15.0,0.0,0.0,5.0,5.0,340.0,5.0,35.0,5.0,70.0,125.0,675.0,155.0,10.0,0.0,15.0,0.0,30.0,30.0,295.0,0.0,0.0,5.0,460.0,40.0,0.0,0.0,40.0,5.0,100.0,85.0,0.0,15.0,1055.0,260.0,5.0,65.0,0.0,105.0,0.0,5.0,235.0,180.0,0.0,0.0,160.0,0.0,0.0,40.0,480.0,90.0,305.0
2017-03,10.0,95.0,20.0,20.0,0.0,0.0,0.0,60.0,335.0,215.0,20.0,960.0,495.0,1190.0,55.0,15.0,30.0,0.0,0.0,80.0,230.0,5.0,0.0,0.0,0.0,0.0,585.0,15.0,45.0,10.0,160.0,160.0,835.0,315.0,65.0,5.0,5.0,0.0,35.0,30.0,600.0,0.0,0.0,10.0,735.0,45.0,0.0,0.0,95.0,10.0,180.0,85.0,0.0,15.0,1260.0,290.0,0.0,90.0,5.0,280.0,0.0,0.0,515.0,200.0,0.0,0.0,375.0,0.0,5.0,25.0,765.0,70.0,845.0
2017-04,0.0,45.0,10.0,5.0,0.0,0.0,5.0,45.0,380.0,230.0,0.0,850.0,520.0,1170.0,85.0,5.0,25.0,0.0,10.0,65.0,75.0,0.0,5.0,0.0,35.0,0.0,555.0,5.0,45.0,10.0,235.0,95.0,930.0,215.0,100.0,20.0,0.0,0.0,25.0,15.0,475.0,0.0,0.0,0.0,580.0,80.0,5.0,5.0,60.0,5.0,235.0,95.0,5.0,40.0,810.0,240.0,15.0,55.0,10.0,175.0,15.0,0.0,495.0,235.0,0.0,0.0,460.0,0.0,15.0,45.0,690.0,30.0,665.0
2017-05,20.0,15.0,50.0,30.0,5.0,0.0,0.0,85.0,505.0,480.0,5.0,1310.0,955.0,1605.0,155.0,0.0,25.0,0.0,5.0,30.0,130.0,5.0,10.0,0.0,25.0,0.0,1160.0,20.0,140.0,35.0,300.0,80.0,1295.0,460.0,110.0,15.0,20.0,5.0,15.0,30.0,605.0,0.0,0.0,10.0,1285.0,90.0,5.0,0.0,95.0,15.0,320.0,105.0,15.0,45.0,1125.0,215.0,30.0,135.0,0.0,300.0,5.0,0.0,670.0,325.0,0.0,0.0,605.0,0.0,15.0,35.0,980.0,40.0,1295.0
2017-06,5.0,5.0,10.0,20.0,0.0,0.0,10.0,55.0,505.0,355.0,5.0,1160.0,750.0,1620.0,75.0,5.0,20.0,0.0,5.0

In [ ]:
growth = growingcate.pct_change()

In [ ]:
growth = growth.replace([np.inf, -np.inf], np.nan)

In [ ]:
growth = growth.fillna(0)

In [ ]:
growth

product_category_name,Agro Industria E Comercio,Alimentos,Alimentos Bebidas,Artes,Artes E Artesanato,Artigos De Festas,Artigos De Natal,Audio,Automotivo,Bebes,Bebidas,Beleza Saude,Brinquedos,Cama Mesa Banho,Casa Conforto,Casa Conforto 2,Casa Construcao,Cds Dvds Musicais,Cine Foto,Climatizacao,Consoles Games,Construcao Ferramentas Construcao,Construcao Ferramentas Ferramentas,Construcao Ferramentas Iluminacao,Construcao Ferramentas Jardim,Construcao Ferramentas Seguranca,Cool Stuff,Dvds Blu Ray,Eletrodomesticos,Eletrodomesticos 2,Eletronicos,Eletroportateis,Esporte Lazer,Fashion Bolsas E Acessorios,Fashion Calcados,Fashion Esporte,Fashion Roupa Feminina,Fashion Roupa Infanto Juvenil,Fashion Roupa Masculina,Fashion Underwear E Moda Praia,Ferramentas Jardim,Flores,Fraldas Higiene,Industria Comercio E Negocios,Informatica Acessorios,Instrumentos Musicais,La Cuisine,Livros Importados,Livros Interesse Geral,Livros Tecnicos,Malas Acessorios,Market Place,Moveis Colchao E Estofado,Moveis Cozinha Area De Servico Jantar E Jardim,Moveis Decoracao,Moveis Escritorio,Moveis Quarto,Moveis Sala,Musica,Papelaria,Pc Gamer,Pcs,Perfumaria,Pet Shop,Portateis Casa Forno E Cafe,Portateis Cozinha E Preparadores De Alimentos,Relogios Presentes,Seguros E Servicos,Sinalizacao E Seguranca,Tablets Impressao Imagem,Telefonia,Telefonia Fixa,Utilidades Domesticas
Month_Year,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2016-10,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
2016-12,0.000000,-1.000000,0.000000,0.000000,0.0,0.000000,0.000000,-1.000000,-1.000000,-1.000000,0.000000,-1.000000,-1.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000,-1.000000,0.000000,-1.000000,-0.857143,-1.000000,0.000000,-1.000000,0.0,-1.000000,0.000000,-1.000000,0.000000,-1.000000,-1.000000,-1.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,-1.000000,-1.000000,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,-1.000000,-1.000000,0.000000,0.000000,-1.000000,0.0,0.000000,0.000000,-1.000000,-1.000000,-1.000000
2017-01,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,30.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
2017-02,2.500000,3.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,1.777778,-0.032258,0.000000,0.787500,0.512195,2.255814,0.000000,-1.000000,0.000000,0.000000,0.000000,2.666667,-0.052632,0.000000,0.000000,0.000000,0.000000,0.000000,1.193548,0.000000,0.000000,-0.833333,0.272727,3.166667,1.288136,0.000000,1.000000,0.000000,0.500000,0.0,5.000000,2.000000,0.552632,0.000000,0.000000,0.000000,2.833333,7.000000,0.0,0.000000,3.000000,0.000000,4.000000,2.400000,0.000000,2.000000,0.623077,7.666667,-0.875000,0.625000,-1.00,2.000000,0.000000,0.000000,0.566667,2.000000,0.000000,0.000000,1.909091,-1.0,0.00000

# **Categories with worse reviews**

In [ ]:
worsereviews = masterSeller.groupby('product_category_name', as_index=False).agg(wreview = ('review_score', 'mean'))

In [ ]:
worsereviews['rank'] = worsereviews['wreview'].rank(ascending=False)

In [ ]:
worsereviews.sort_values(by='rank')

,product_category_name,wreview,rank
17,Cds Dvds Musicais,4.666667,1.0
41,Flores,4.538462,2.0
48,Livros Interesse Geral,4.462451,3.0
22,Construcao Ferramentas Ferramentas,4.425532,4.0
49,Livros Tecnicos,4.421260,5.0
...,...,...,...
38,Fashion Roupa Masculina,3.727273,69.0
55,Moveis Escritorio,3.626889,70.0
65,Portateis Cozinha E Preparadores De Alimentos,3.428571,71.0
60,Pc Gamer,3.125000,72.0
